In [1]:
import os
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
SPLITS = ['train', 'valid', 'test']

SPLIT_ALIASES = {
    'train': 'train',
    'valid': 'valid',  # guns_n_knives
    'val':   'valid',  # kaggle weapon-detection
    'test':  'test',
}

def find_dataset_root(base: Path) -> Path | None:
    all_splits = list(SPLIT_ALIASES.keys())
    if any((base / s).exists() for s in all_splits):
        return base
    for child in sorted(base.rglob('*')):
        if child.is_dir() and any((child / s).exists() for s in all_splits):
            return child
    return None

gnk_root    = None
kaggle_root = None

for dataset_dir in sorted(INPUT_ROOT.iterdir()):
    root = find_dataset_root(dataset_dir)
    if root is None:
        continue
    yaml_file = root / 'data.yaml'
    if yaml_file.exists():
        content = yaml_file.read_text()
        if 'gun' in content and 'knife' in content and 'pistol' not in content:
            gnk_root = root
            print(f'✅ guns_n_knives  → {root}')
        elif 'pistol' in content:
            kaggle_root = root
            print(f'✅ weapon-detection → {root}')

if gnk_root is None or kaggle_root is None:
    print('\n Could not auto-detect one or both datasets.')
    print('Available input directories:')
    for d in sorted(INPUT_ROOT.iterdir()):
        print(f'  {d}')
    print('\nSet gnk_root and kaggle_root manually in the cell below.')
else:
    print('\nBoth datasets found — ready to merge!')

✅ guns_n_knives  → /kaggle/input/datasets/datatou/guns-n-knives-computer-vision-model

⚠️  Could not auto-detect one or both datasets.
Available input directories:
  /kaggle/input/datasets

Set gnk_root and kaggle_root manually in the cell below.


In [2]:
gnk_root    = Path('/kaggle/input/datasets/datatou/guns-n-knives-computer-vision-model')
kaggle_root = Path('/kaggle/input/datasets/mehmetcubukcu/weapon-detection/weapon-detection')

print('gnk_root    =', gnk_root)
print('kaggle_root =', kaggle_root)

gnk_root    = /kaggle/input/datasets/datatou/guns-n-knives-computer-vision-model
kaggle_root = /kaggle/input/datasets/mehmetcubukcu/weapon-detection/weapon-detection


In [3]:
import yaml

def read_yaml_names(root: Path):
    yf = root / 'data.yaml'
    if not yf.exists():
        return None
    with open(yf) as f:
        data = yaml.safe_load(f)
    return data.get('names', [])

gnk_names    = read_yaml_names(gnk_root)
kaggle_names = read_yaml_names(kaggle_root)

print('guns_n_knives classes :', gnk_names)
print('Kaggle classes        :', kaggle_names)

for name, root in [('guns_n_knives', gnk_root), ('weapon-detection', kaggle_root)]:
    print(f'\n{name}:')
    for split in SPLIT_ALIASES.keys():
        img_dir = root / split / 'images'
        lbl_dir = root / split / 'labels'
        n_img = len(list(img_dir.glob('*'))) if img_dir.exists() else 0
        n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
        print(f'  {split:6s}: {n_img:>5} images  {n_lbl:>5} labels')

guns_n_knives classes : ['gun', 'knife']
Kaggle classes        : ['pistol', 'smartphone', 'knife', 'monedero', 'billete', 'tarjeta']

guns_n_knives:
  train :  6745 images   6745 labels
  valid :  1933 images   1933 labels
  val   :     0 images      0 labels
  test  :   957 images    957 labels

weapon-detection:
  train :  5002 images   5002 labels
  valid :     0 images      0 labels
  val   :   450 images    450 labels
  test  :   407 images    407 labels


In [4]:
import shutil

GNK_CLASS_MAP = {0: 0, 1: 1}

KAGGLE_CLASS_MAP = {
    0: 0,   # pistol      -> gun
    1: 4,   # smartphone  -> smartphone
    2: 1,   # knife       -> knife
    3: 5,   # monedero    -> purse
    4: 3,   # billete     -> bill
    5: 2,   # tarjeta     -> card
}

FINAL_CLASSES = ['gun', 'knife', 'card', 'bill', 'smartphone', 'purse']
IMG_EXTS      = {'.jpg', '.jpeg', '.png', '.bmp'}
OUT_ROOT      = Path('/kaggle/working/merged_dataset')


def remap_label_file(src: Path, dst: Path, class_map: dict) -> int:
    """Rewrite a YOLO .txt label with remapped class IDs. Returns line count."""
    lines_out = []
    if src.exists():
        for line in src.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            old_cls = int(parts[0])
            if old_cls not in class_map:
                print(f'  [WARN] Unknown class {old_cls} in {src.name} — skipped')
                continue
            parts[0] = str(class_map[old_cls])
            lines_out.append(' '.join(parts))
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text('\n'.join(lines_out) + ('\n' if lines_out else ''))
    return len(lines_out)


def merge_split(src_split: str, canonical_split: str) -> dict:
    stats = {'gnk_img': 0, 'gnk_ann': 0, 'kag_img': 0, 'kag_ann': 0}
    out_img = OUT_ROOT / canonical_split / 'images'
    out_lbl = OUT_ROOT / canonical_split / 'labels'
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    for prefix, src_root, class_map, img_key, ann_key in [
        ('gnk',    gnk_root,    GNK_CLASS_MAP,    'gnk_img', 'gnk_ann'),
        ('kaggle', kaggle_root, KAGGLE_CLASS_MAP,  'kag_img', 'kag_ann'),
    ]:
        img_dir = src_root / src_split / 'images'
        lbl_dir = src_root / src_split / 'labels'
        if not img_dir.exists():
            print(f'  [{prefix}] {src_split} images dir not found — skipping')
            continue
        for img in sorted(img_dir.glob('*')):
            if img.suffix.lower() not in IMG_EXTS:
                continue
            new_name = f'{prefix}_{img.name}'
            shutil.copy2(img, out_img / new_name)
            stats[img_key] += 1
            src_lbl = lbl_dir / img.with_suffix('.txt').name
            dst_lbl = out_lbl / (new_name.rsplit('.', 1)[0] + '.txt')
            stats[ann_key] += remap_label_file(src_lbl, dst_lbl, class_map)

    return stats


total = {'gnk_img': 0, 'gnk_ann': 0, 'kag_img': 0, 'kag_ann': 0}

for src_split, canonical_split in SPLIT_ALIASES.items():
    print(f'\nMerging [{src_split}] -> [{canonical_split}] ...')
    s = merge_split(src_split, canonical_split)
    for k in total: total[k] += s[k]
    print(f'  guns_n_knives  : {s["gnk_img"]:>5} images   {s["gnk_ann"]:>6} annotations')
    print(f'  weapon-detect  : {s["kag_img"]:>5} images   {s["kag_ann"]:>6} annotations')

print(f'\n{"="*55}')
print(f'  TOTAL  images : {total["gnk_img"] + total["kag_img"]}')
print(f'  TOTAL  annots : {total["gnk_ann"] + total["kag_ann"]}')
print(f'{"="*55}')


Merging [train] -> [train] ...
  guns_n_knives  :  6745 images     9666 annotations
  weapon-detect  :  5002 images     5002 annotations

Merging [valid] -> [valid] ...
  [kaggle] valid images dir not found — skipping
  guns_n_knives  :  1933 images     2766 annotations
  weapon-detect  :     0 images        0 annotations

Merging [val] -> [valid] ...
  [gnk] val images dir not found — skipping
  guns_n_knives  :     0 images        0 annotations
  weapon-detect  :   450 images      450 annotations

Merging [test] -> [test] ...
  guns_n_knives  :   957 images     1370 annotations
  weapon-detect  :   407 images      407 annotations

  TOTAL  images : 15494
  TOTAL  annots : 19661


In [5]:
yaml_content = f"""# Merged Weapon Detection Dataset — YOLOv8
# Sources:
#   1. guns_n_knives  (Roboflow, CC BY 4.0)
#   2. mehmetcubukcu/weapon-detection  (Kaggle)
#
# Class remapping:
#   guns_n_knives : gun(0)->gun   knife(1)->knife
#   Kaggle        : pistol(0)->gun        smartphone(1)->smartphone
#                   knife(2)->knife       monedero(3)->purse
#                   billete(4)->bill      tarjeta(5)->card

path: {OUT_ROOT}
train: train/images
val:   valid/images
test:  test/images

nc: {len(FINAL_CLASSES)}
names: {FINAL_CLASSES}
"""

(OUT_ROOT / 'data.yaml').write_text(yaml_content)
print('data.yaml written:')
print(yaml_content)

data.yaml written:
# Merged Weapon Detection Dataset — YOLOv8
# Sources:
#   1. guns_n_knives  (Roboflow, CC BY 4.0)
#   2. mehmetcubukcu/weapon-detection  (Kaggle)
#
# Class remapping:
#   guns_n_knives : gun(0)->gun   knife(1)->knife
#   Kaggle        : pistol(0)->gun        smartphone(1)->smartphone
#                   knife(2)->knife       monedero(3)->purse
#                   billete(4)->bill      tarjeta(5)->card

path: /kaggle/working/merged_dataset
train: train/images
val:   valid/images
test:  test/images

nc: 6
names: ['gun', 'knife', 'card', 'bill', 'smartphone', 'purse']



In [6]:
import zipfile

ZIP_PATH = Path('/kaggle/working/merged_dataset.zip')

print(f'Zipping {OUT_ROOT} → {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(OUT_ROOT.rglob('*')):
        if file.is_file():
            zf.write(file, file.relative_to(OUT_ROOT.parent))

size_mb = ZIP_PATH.stat().st_size / 1_048_576
print(f'Done — {ZIP_PATH.name}  ({size_mb:.1f} MB)')
print('\nDownload it from the Kaggle Output panel on the right →')

Zipping /kaggle/working/merged_dataset → /kaggle/working/merged_dataset.zip ...
Done — merged_dataset.zip  (2167.4 MB)

Download it from the Kaggle Output panel on the right →


In [7]:
!pip install ultralytics -q
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data  = str(OUT_ROOT / 'data.yaml'),
    epochs = 70,
    imgsz  = 640,
    batch  = 16,
    name   = 'weapon_detector',
)
print('Training cell ready — uncomment to run.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.1 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/merged_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0

In [8]:
#model = YOLO('/kaggle/working/runs/detect/weapon_detector/weights/best.pt') 

#metrics = model.val()